# TurboPairFormer — Quickstart

[**TurboPairFormer**](https://pypi.org/project/turbopairformer/) — developed by
[Lambda](https://lambda.ai), [Stevens Institute of Technology](https://www.stevens.edu), and
[OpenFold](https://openfold.io) — provides standalone CUDA kernels for the two most expensive
operations in an [AlphaFold3](https://www.nature.com/articles/s41586-024-07487-w)-style pair stack:

| Operation | Function | What it replaces |
|---|---|---|
| Triangle Attention | `turbo_attention` | Row/column-wise attention over an `N x N` pair representation |
| Triangle Multiplicative Update | `turbo_trimul` | The outgoing / incoming "TriMul" pair update |

Both are ordinary autograd-aware functions. They do **not** require OpenFold3 — you pass raw
tensors (and, for TriMul, your own module's weights) and get gradients back.

This notebook covers install, the two operator calls, a numerical check against a reference
PyTorch implementation, and a speed / memory benchmark against stock PyTorch.

> Every number below was produced by executing this notebook on a single [NVIDIA H100 80GB HBM3](https://www.nvidia.com/en-us/data-center/h100/).

## Background

AlphaFold3 predicts the structure of biomolecular complexes by refining a **pair
representation**: a table holding one feature vector for every pair of residues in the input.
The stack that refines it is the **Pairformer**.

Each Pairformer block applies two operations. Both enforce the same idea — what the model
believes about two residues should stay consistent with what it believes about each of them
and any third residue. Hence "triangle".

- **Triangle multiplicative update** (outgoing / incoming) — updates every pair by combining
  it with every possible third residue.
- **Triangle attention** (starting / ending node) — lets each pair attend along its own row
  or column of the table, guided by the pair features themselves.

Both scale with the cube of the sequence length, and together they dominate the trunk's time
and memory. Triangle attention builds a score tensor with an entry for every triple of
residues, per head — for a thousand-residue sequence that is over 60 GB of activations in
stock PyTorch.

**TurboPairFormer** replaces both with fused CUDA kernels for Hopper GPUs, Apache-2.0 licensed
and built for **training**: the backward pass is fused, gradients reach every input including
the pair bias, and a DDP consistency check ships with it. The module contract follows
[OpenFold3](https://github.com/aqlaboratory/openfold-3), the open AlphaFold3 reproduction, but nothing here imports it.

## 1. Install

TurboPairFormer 0.1.0 ships one ahead-of-time compiled wheel. The build is pinned, and the
loader verifies the ABI and binary hashes before loading an extension rather than silently
recompiling — so the environment has to match:

- Linux x86-64, glibc ≥ 2.28
- CPython **3.14**
- PyTorch **2.10.0** with the CUDA **12.8** runtime
- NVIDIA **H100** or **H200** (Hopper, compute capability 9.0)

Only the first two are yours to arrange. `turbopairformer` pins `torch==2.10.0`, and the
PyPI wheel for that version is already the CUDA 12.8 build, so pip resolves the rest:

```bash
conda create -n turbopairformer python=3.14 pip -y
conda activate turbopairformer

# Put the environment's own pip first, so the install lands in this env.
export PATH="$CONDA_PREFIX/bin:$PATH"
hash -r

pip install turbopairformer
```

Start this notebook's kernel from that environment.

No CUDA toolkit is needed to *run* the wheel — only to build one; an NVIDIA driver is
enough.

In [1]:
import os

# Sweeping many sequence lengths in one process fragments the allocator, which can
# make the stock baseline OOM at a size it otherwise fits. Must precede CUDA init.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import torch
import turbopairformer

capability = torch.cuda.get_device_capability(0)

print(f"turbopairformer  {turbopairformer.__version__}")
print(f"torch            {torch.__version__}  (CUDA {torch.version.cuda})")
print(f"GPU              {torch.cuda.get_device_name(0)}  (sm_{capability[0]}{capability[1]})")

assert capability == (9, 0), "TurboPairFormer 0.1.0 ships sm_90a kernels: H100 / H200 only"

turbopairformer  0.1.0
torch            2.10.0+cu128  (CUDA 12.8)
GPU              NVIDIA H100 80GB HBM3  (sm_90)


## 2. Triangle attention

`turbo_attention(q, k, v, res_mask=None, pair_bias=...)` runs a full triangle-attention
forward pass. The pair bias is **required**; the mask is optional and defaults to "every
position is valid", so it is usually passed by keyword.

In [2]:
from turbopairformer import turbo_attention

torch.manual_seed(0)
B, N, H, D = 1, 128, 4, 32  # batch, sequence, heads, head dimension

# q / k / v: [batch, outer, heads, sequence, head_dim]. For triangle attention the
# "outer" axis is the pair row (or column) being attended within, so outer == N.
q, k, v = [
    torch.randn(B, N, H, N, D, device="cuda", dtype=torch.bfloat16, requires_grad=True)
    for _ in range(3)
]

# Pair bias: [batch, 1, heads, sequence, sequence] — shared across the outer axis.
pair_bias = torch.randn(B, 1, H, N, N, device="cuda", dtype=torch.float32, requires_grad=True)

out = turbo_attention(q, k, v, pair_bias=pair_bias)
out.float().square().mean().backward()

print(f"output      {tuple(out.shape)}  {out.dtype}")
for name, tensor in (("dq", q), ("dk", k), ("dv", v)):
    print(f"{name:<12}{tuple(tensor.grad.shape)}  {tensor.grad.dtype}")
print(f"dpair_bias  {tuple(pair_bias.grad.shape)}  {pair_bias.grad.dtype}")

output      (1, 128, 128, 4, 32)  torch.bfloat16
dq          (1, 128, 4, 128, 32)  torch.bfloat16
dk          (1, 128, 4, 128, 32)  torch.bfloat16
dv          (1, 128, 4, 128, 32)  torch.bfloat16
dpair_bias  (1, 1, 4, 128, 128)  torch.float32


Two layout details worth pinning down, because they are the most common source of silent
shape bugs:

| Tensor | Shape |
|---|---|
| `q`, `k`, `v` (in) | `[batch, outer, heads, sequence, head_dim]` |
| `pair_bias` (in) | `[batch, 1, heads, sequence, sequence]` |
| `res_mask` (in, optional) | `[batch, outer, 1, 1, sequence]`, additive: `0.0` valid, `-inf` padded |
| return value (out) | `[batch, outer, sequence, heads, head_dim]` |

The output swaps the `heads` and `sequence` axes relative to the input, which lands it in
exactly the layout a gating projection wants (`.flatten(-2)` gives you `heads * head_dim`).

The kernel applies the `1/sqrt(head_dim)` softmax scale itself — do **not** pre-scale `q`.
`pair_bias` may be FP32 or BF16: FP32 is cast at the kernel boundary and autograd casts the
gradient back, so an FP32 bias parameter receives an FP32 gradient.

## 3. Triangle multiplicative update (TriMul)

`turbo_trimul(module, z, mask=None)` takes *your* module and reads its weights directly.
The module must expose the OpenFold3 triangle-multiplication weight contract — six
bias-free `nn.Linear` layers, two `LayerNorm`s, `c_z` / `c_hidden`, and an explicit
`_outgoing` bool. Importing OpenFold3 is not required.

In [3]:
from torch import nn
from turbopairformer import turbo_trimul


class TriangleMultiplication(nn.Module):
    '''Minimal module matching the weight contract turbo_trimul expects.'''

    def __init__(self, channels: int, outgoing: bool):
        super().__init__()
        self.c_z = channels
        self.c_hidden = channels          # the fast path requires c_z == c_hidden
        self._outgoing = outgoing         # must be an explicit bool

        self.layer_norm_in = nn.LayerNorm(channels, eps=1e-5)
        self.layer_norm_out = nn.LayerNorm(channels, eps=1e-5)
        self.linear_a_p = nn.Linear(channels, channels, bias=False)
        self.linear_a_g = nn.Linear(channels, channels, bias=False)
        self.linear_b_p = nn.Linear(channels, channels, bias=False)
        self.linear_b_g = nn.Linear(channels, channels, bias=False)
        self.linear_g = nn.Linear(channels, channels, bias=False)
        self.linear_z = nn.Linear(channels, channels, bias=False)

    def forward(self, z, mask=None):
        return turbo_trimul(self, z, mask)


torch.manual_seed(0)
N, C = 256, 128

# Weights stay FP32; the pair tensor is BF16.
trimul_out = TriangleMultiplication(C, outgoing=True).cuda()
z = torch.randn(1, N, N, C, device="cuda", dtype=torch.bfloat16, requires_grad=True)

z_out = trimul_out(z)
z_out.float().square().mean().backward()

print(f"z      {tuple(z.shape)}  {z.dtype}")
print(f"output {tuple(z_out.shape)}  {z_out.dtype}")
print(f"dz     {tuple(z.grad.shape)}")
print(f"dW     {tuple(trimul_out.linear_a_p.weight.grad.shape)}")

z      (1, 256, 256, 128)  torch.bfloat16
output (1, 256, 256, 128)  torch.bfloat16
dz     (1, 256, 256, 128)
dW     (128, 128)


### Fail closed, with a reason

The facade validates the whole contract *before* allocating anything or loading a CUDA
extension. `check_trimul_support` gives you the same verdict without running the kernel,
which is what you want when deciding whether to route a given site to the fast path.

In [4]:
from turbopairformer import check_trimul_support, TRIMUL_STOCK_FALLBACK_THRESHOLD

print(f"stock fallback threshold: N > {TRIMUL_STOCK_FALLBACK_THRESHOLD}\n")

# Supported.
print(check_trimul_support(trimul_out, z, None).reason)

# Too short — short TriMul sites stay on the stock PyTorch route by design.
short_z = torch.randn(1, 64, 64, C, device="cuda", dtype=torch.bfloat16)
print(check_trimul_support(trimul_out, short_z, None).reason)

# Wrong parameter dtype.
bf16_module = TriangleMultiplication(C, outgoing=True).cuda().to(torch.bfloat16)
print(check_trimul_support(bf16_module, z, None).reason)

# Calling anyway raises instead of degrading silently.
try:
    turbo_trimul(bf16_module, z)
except ValueError as error:
    print(f"\nValueError: {error}")

stock fallback threshold: N > 100

supported Hopper sm90a BF16 TriMul fast path: batch=1, N=256, C_z=C_hidden=128
N=64 is at or below the fixed stock PyTorch fallback threshold 100; TurboPairFormer TriMul AOT starts at N=101
linear_a_p.weight must have dtype torch.float32

ValueError: TurboPairFormer TriMul unsupported [parameter_dtype]: linear_a_p.weight must have dtype torch.float32


## 4. Numerical correctness

Comparing two BF16 implementations only shows that they round alike; it cannot show whether
either is close to the true value. The reference below is an **FP32** implementation of the
same math, and both the kernel and stock BF16 PyTorch are measured against it.

Triangle attention first. One case covers it: starting-node and ending-node attention
are the same kernel call on a transposed pair tensor, not separate code paths.


In [5]:
def reference_attention(q, k, v, pair_bias, dtype, res_mask=None):
    '''Textbook triangle attention. dtype=float32 is the ground truth;
    dtype=bfloat16 is what a stock PyTorch model actually runs.
    res_mask is additive: 0.0 for valid keys, -inf for padding.'''
    if res_mask is not None:
        pair_bias = pair_bias.float() + res_mask
    scale = q.shape[-1] ** -0.5
    q, k, v, pair_bias = (t.to(dtype) for t in (q, k, v, pair_bias))
    scores = torch.matmul(q, k.transpose(-1, -2)) * scale + pair_bias
    probabilities = torch.softmax(scores, dim=-1, dtype=torch.float32).to(dtype)
    return torch.matmul(probabilities, v).permute(0, 1, 3, 2, 4)


def relative_error(actual, expected):
    return ((actual.float() - expected.float()).abs().max()
            / expected.float().abs().max()).item()


torch.manual_seed(0)
B, N, H, D = 1, 128, 4, 32
q, k, v = [
    torch.randn(B, N, H, N, D, device="cuda", dtype=torch.bfloat16, requires_grad=True)
    for _ in range(3)
]
pair_bias = torch.randn(B, 1, H, N, N, device="cuda", dtype=torch.bfloat16, requires_grad=True)

truth = reference_attention(q, k, v, pair_bias, torch.float32)
stock = reference_attention(q, k, v, pair_bias, torch.bfloat16)
turbo = turbo_attention(q, k, v, pair_bias=pair_bias)

grad_seed = torch.randn_like(truth)
inputs = [q, k, v, pair_bias]
truth_grads = torch.autograd.grad(truth, inputs, grad_seed, retain_graph=True)
stock_grads = torch.autograd.grad(stock, inputs, grad_seed, retain_graph=True)
turbo_grads = torch.autograd.grad(turbo.float(), inputs, grad_seed)

print(f"{'tensor':<12}{'stock bf16':>14}{'turbo':>14}   (max rel. error vs FP32)")
print("-" * 54)
print(f"{'output':<12}{relative_error(stock, truth):>14.2e}{relative_error(turbo, truth):>14.2e}")
for name, s, t, r in zip(["dq", "dk", "dv", "dpair_bias"], stock_grads, turbo_grads, truth_grads):
    print(f"{name:<12}{relative_error(s, r):>14.2e}{relative_error(t, r):>14.2e}")

tensor          stock bf16         turbo   (max rel. error vs FP32)
------------------------------------------------------
output            9.96e-03      2.88e-03
dq                1.23e-02      6.13e-03
dk                8.57e-03      5.71e-03
dv                1.06e-02      5.32e-03
dpair_bias        6.80e-03      6.80e-03


The same comparison for TriMul, in both directions — outgoing and incoming are separate
kernel paths, so each needs its own check.

In [6]:
def reference_trimul(module, z, dtype, mask=None):
    '''Textbook triangle multiplicative update.
    mask is multiplicative over pairs: 1 valid, 0 padded.'''
    import torch.nn.functional as F

    weight = lambda layer: layer.weight.to(dtype)
    z_normed = module.layer_norm_in(z.float()).to(dtype)

    a = torch.sigmoid(F.linear(z_normed, weight(module.linear_a_g))) * \
        F.linear(z_normed, weight(module.linear_a_p))
    b = torch.sigmoid(F.linear(z_normed, weight(module.linear_b_g))) * \
        F.linear(z_normed, weight(module.linear_b_p))

    if mask is not None:
        pair_gate = mask.unsqueeze(-1).to(dtype)
        a = a * pair_gate
        b = b * pair_gate

    equation = "...ikc,...jkc->...ijc" if module._outgoing else "...kic,...kjc->...ijc"
    x = torch.einsum(equation, a, b)
    x = F.linear(module.layer_norm_out(x.float()).to(dtype), weight(module.linear_z))
    return torch.sigmoid(F.linear(z_normed, weight(module.linear_g))) * x


print(f"{'direction':<12}{'stock bf16':>14}{'turbo':>14}   (max rel. error vs FP32)")
print("-" * 54)
for outgoing in (True, False):
    torch.manual_seed(0)
    module = TriangleMultiplication(128, outgoing=outgoing).cuda()
    z = torch.randn(1, 256, 256, 128, device="cuda", dtype=torch.bfloat16, requires_grad=True)

    truth = reference_trimul(module, z, torch.float32)
    stock = reference_trimul(module, z, torch.bfloat16)
    turbo = turbo_trimul(module, z)

    label = "outgoing" if outgoing else "incoming"
    print(f"{label:<12}{relative_error(stock, truth):>14.2e}{relative_error(turbo, truth):>14.2e}")

direction       stock bf16         turbo   (max rel. error vs FP32)
------------------------------------------------------


outgoing          7.19e-03      5.68e-03


incoming          7.67e-03      6.15e-03


The kernel sits at the same BF16 noise floor as stock PyTorch — it is not trading accuracy
for speed.

## 5. Speed and memory

Triangle attention materialises a score tensor shaped `[batch, outer, heads, N, N]` — one
entry per triple of residues, per head. For a thousand-residue sequence with four heads
that is 8 GB in BF16, plus another 16 GB for the FP32 softmax over it.

TurboPairFormer fuses the forward with an online softmax and recomputes the backward with
zero atomics, so the score tensor is never written to memory. The peak-memory column below
is the evidence: stock needs 65 GB for a forward pass at that size, the kernel needs 1.9 GB
— the difference is the score tensor that one of them never stores.

In [7]:
import gc
import time
from typing import NamedTuple


class Measurement(NamedTuple):
    '''One timed configuration: wall-clock time and peak GPU memory.'''

    milliseconds: float
    peak_gb: float


def release(*names):
    '''Drop notebook-scope tensors so a benchmark starts from a clean GPU.'''
    for name in names:
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()


def benchmark(fn, backward, iterations=10):
    '''Mean wall-clock time over `iterations`, plus peak allocated memory.
    Returns None when the configuration does not fit in GPU memory.'''
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        for _ in range(3):  # warm up
            output = fn()
            if backward:
                output.float().square().mean().backward()
        torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(iterations):
            output = fn()
            if backward:
                output.float().square().mean().backward()
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        output = None
        gc.collect()
        torch.cuda.empty_cache()
        return None

    milliseconds = (time.perf_counter() - start) / iterations * 1e3
    return Measurement(milliseconds, torch.cuda.max_memory_allocated() / 2**30)


def report(rows, header):
    print(header)
    print(f"{'N':>6}{'stock ms':>11}{'turbo ms':>11}{'speedup':>10}"
          f"{'stock GB':>11}{'turbo GB':>11}{'memory':>9}")
    print("-" * 69)
    for sequence, stock, turbo in rows:
        if stock is None:  # stock PyTorch ran out of memory; the kernel did not
            print(f"{sequence:>6}{'OOM':>11}{turbo.milliseconds:>11.2f}{'--':>10}"
                  f"{'OOM':>11}{turbo.peak_gb:>11.2f}{'--':>9}")
            continue
        print(f"{sequence:>6}{stock.milliseconds:>11.2f}{turbo.milliseconds:>11.2f}"
              f"{stock.milliseconds / turbo.milliseconds:>9.2f}x"
              f"{stock.peak_gb:>11.2f}{turbo.peak_gb:>11.2f}"
              f"{stock.peak_gb / turbo.peak_gb:>8.2f}x")

Triangle attention against stock PyTorch, swept from 128 to 1024 residues — forward
only, then forward and backward.

In [8]:
def stock_attention(q, k, v, pair_bias):
    return reference_attention(q, k, v, pair_bias, torch.bfloat16)


# The correctness cells above still hold tensors and autograd graphs; at N = 1024
# stock triangle attention needs most of an 80GB card, so start from a clean slate.
release("out", "z", "z_out", "short_z", "bf16_module", "module", "masked", "expected",
        "truth", "stock", "turbo", "grad_seed", "inputs",
        "truth_grads", "stock_grads", "turbo_grads", "q", "k", "v", "pair_bias")

H, D = 4, 32
forward_rows, training_rows = [], []

for N in (128, 256, 512, 768, 1024):
    torch.manual_seed(0)
    q, k, v = [
        torch.randn(1, N, H, N, D, device="cuda", dtype=torch.bfloat16, requires_grad=True)
        for _ in range(3)
    ]
    pair_bias = torch.randn(1, 1, H, N, N, device="cuda", dtype=torch.bfloat16,
                            requires_grad=True)

    for backward, rows in ((False, forward_rows), (True, training_rows)):
        stock = benchmark(lambda: stock_attention(q, k, v, pair_bias), backward)
        turbo = benchmark(lambda: turbo_attention(q, k, v, pair_bias=pair_bias), backward)
        rows.append((N, stock, turbo))

    del q, k, v, pair_bias
    gc.collect()
    torch.cuda.empty_cache()

report(forward_rows, "Triangle attention — forward only (batch 1, 4 heads, head_dim 32)\n")
print()
report(training_rows, "Triangle attention — forward + backward\n")

Triangle attention — forward only (batch 1, 4 heads, head_dim 32)

     N   stock ms   turbo ms   speedup   stock GB   turbo GB   memory
---------------------------------------------------------------------
   128       0.14       0.12     1.20x       0.21       0.10    2.13x
   256       0.92       0.18     5.00x       1.13       0.18    6.17x
   512       7.01       0.99     7.09x       8.32       0.52   15.99x
   768      23.55       3.02     7.79x      27.64       1.08   25.54x
  1024      64.41       6.81     9.46x      65.08       1.87   34.83x

Triangle attention — forward + backward

     N   stock ms   turbo ms   speedup   stock GB   turbo GB   memory
---------------------------------------------------------------------
   128       0.61       0.54     1.13x       0.22       0.14    1.57x
   256       2.51       0.89     2.81x       1.18       0.36    3.31x
   512      17.69       4.68     3.78x       8.51       1.59    5.36x
   768      57.97      14.08     4.12x      28.06  

The same sweep for TriMul, outgoing direction, at 128 channels.

In [9]:
release("q", "k", "v", "pair_bias", "stock", "turbo")
forward_rows, training_rows = [], []

for N in (128, 256, 512, 768, 1024):
    torch.manual_seed(0)
    module = TriangleMultiplication(128, outgoing=True).cuda()
    z = torch.randn(1, N, N, 128, device="cuda", dtype=torch.bfloat16, requires_grad=True)

    for backward, rows in ((False, forward_rows), (True, training_rows)):
        stock = benchmark(lambda: reference_trimul(module, z, torch.bfloat16), backward)
        turbo = benchmark(lambda: turbo_trimul(module, z), backward)
        rows.append((N, stock, turbo))

    del module, z
    gc.collect()
    torch.cuda.empty_cache()

report(forward_rows, "TriMul (outgoing) — forward only (batch 1, C_z = C_hidden = 128)\n")
print()
report(training_rows, "TriMul (outgoing) — forward + backward")

TriMul (outgoing) — forward only (batch 1, C_z = C_hidden = 128)

     N   stock ms   turbo ms   speedup   stock GB   turbo GB   memory
---------------------------------------------------------------------
   128       0.42       0.39     1.07x       0.20       0.13    1.51x
   256       0.69       0.38     1.82x       0.57       0.31    1.86x
   512       2.89       0.72     4.03x       2.08       1.02    2.04x
   768       9.39       1.55     6.06x       4.59       2.20    2.08x
  1024      18.24       2.78     6.57x       8.10       3.86    2.10x

TriMul (outgoing) — forward + backward
     N   stock ms   turbo ms   speedup   stock GB   turbo GB   memory
---------------------------------------------------------------------
   128       1.84       1.56     1.18x       0.18       0.16    1.12x
   256       2.96       1.33     2.22x       0.50       0.42    1.18x
   512      10.77       3.07     3.51x       1.76       1.45    1.21x
   768      41.42       6.58     6.30x       3.88     

Speedups grow with `N`, which is exactly where a folding model spends its time: the
advantage is small at `N = 128` and is largest on the long sequences that dominate a
training step. The memory column matters just as much — at `N = 1024` the stock triangle
attention backward needs tens of GB for activations that the fused kernel never writes.

## 6. Padded inputs

Real batches are padded: a 100-residue chain in a 128-slot tensor leaves 28 slots holding
nothing meaningful. Both operators reduce over that axis — attention sums over every key,
TriMul over every third residue — so unmasked padding does not merely produce junk where
the padding is, it contaminates the real positions too. A mask tells the kernel which slots
to ignore, and the two operators want it in different forms:

- **Attention** — an *additive* FP32 mask of shape `[batch, outer, 1, 1, sequence]`,
  `0.0` for valid keys and `-inf` for padding. It is added to the scores before the
  softmax, so a padded key gets `exp(-inf) = 0` weight and drops out of the normalisation
  entirely, handing its share back to the real keys.
- **TriMul** — a *multiplicative* pair mask of shape `[batch, N, N]`, `1` valid / `0` padded
  (bool, BF16 or FP32). There is no softmax here, so zeroing the contribution is enough.

Below, 128 attention slots hold 100 real residues and 256 pair slots hold 200. Both
operators are checked against an FP32 reference handed the same mask.

In [10]:
torch.manual_seed(0)
B, N, H, D, valid = 1, 128, 4, 32, 100  # 128 allocated positions, 100 real tokens

q, k, v = [
    torch.randn(B, N, H, N, D, device="cuda", dtype=torch.bfloat16, requires_grad=True)
    for _ in range(3)
]
pair_bias = torch.randn(B, 1, H, N, N, device="cuda", dtype=torch.bfloat16, requires_grad=True)

res_mask = torch.zeros(B, N, 1, 1, N, device="cuda", dtype=torch.float32)
res_mask[..., valid:] = float("-inf")

masked = turbo_attention(q, k, v, res_mask, pair_bias)
expected = reference_attention(q, k, v, pair_bias, torch.float32, res_mask=res_mask)
print(f"attention with padding: max rel. error vs FP32 = "
      f"{relative_error(masked, expected):.2e}")

pair_mask = torch.zeros(1, 256, 256, device="cuda", dtype=torch.bfloat16)
pair_mask[:, :200, :200] = 1.0
module = TriangleMultiplication(128, outgoing=True).cuda()
z = torch.randn(1, 256, 256, 128, device="cuda", dtype=torch.bfloat16, requires_grad=True)

masked_z = turbo_trimul(module, z, pair_mask)
expected_z = reference_trimul(module, z, torch.float32, mask=pair_mask)
print(f"trimul with padding:    max rel. error vs FP32 = "
      f"{relative_error(masked_z, expected_z):.2e}")


attention with padding: max rel. error vs FP32 = 2.58e-03
trimul with padding:    max rel. error vs FP32 = 5.92e-03


## Supported range

| | |
|---|---|
| GPU | H100 / H200 (Hopper, compute capability 9.0), one `sm_90a` wheel |
| dtype | Q/K/V BF16; pair bias BF16 or FP32; TriMul weights FP32, `z` BF16 |
| head dimension | `1 <= D <= 64`; multiples of 8 run natively, others zero-pad |
| TriMul channels | `c_z == c_hidden` in `{32, 64, 96, 128}`, `N > 100` |
| batch prefix | any non-empty broadcastable prefix, flattened into one CUDA batch |

Anything outside this range raises before allocation with a reason string — routing a
rejected site to a stock PyTorch fallback is the host model's job, not the kernel's.

## Links

- PyPI: <https://pypi.org/project/turbopairformer/>
- Reference architecture: [OpenFold3 Pairformer](https://github.com/aqlaboratory/openfold-3/blob/main/openfold3/core/model/latent/pairformer.py)